<a href="https://colab.research.google.com/github/ouchn2580201251107/04399_Big_Data_Analytics_Mining_Tech/blob/main/04399_BigData_Ex6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

案例数据来源于江苏镇江扬中市的高新区企业历史近2年的用电量，希望能够根据历史数据去精准预测未来一个月每一天的用电量，这是一个很典型的回归类问题，和我们的流量预测非常相关，我们来看看如何用数据驱动的方式去完成这样一个预测。

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

%matplotlib inline

Mounted at /content/drive


### 载入数据集合并

In [2]:
_df = pd.read_csv("/content/drive/MyDrive/04399_Big_Data_Analytics&Mining_Tech/data/zhenjiang_power.csv")
df_201609 = pd.read_csv("/content/drive/MyDrive/04399_Big_Data_Analytics&Mining_Tech/data/zhenjiang_power_9.csv")
train_df = pd.concat([_df, df_201609], sort=False)

### 数据一览
数据会被读取成Excel一样行与列的形式。其中每一行是一个样本(一条历史记录)，每一列可以是一个指代特征(对最后结果有贡献的数据因素/维度)

In [3]:
train_df.head(5)

,user_id,record_date,power_consumption
0,1,2015-01-01,1135.0
1,1,2015-01-02,570.0
2,1,2015-01-03,3418.0
3,1,2015-01-04,3968.0
4,1,2015-01-05,3986.0


每一行是一条用电记录(一个企业一天一条记录)，每一列是不同的数据维度信息，第一列是用电量，第二列是日期，第三列是企业id

### 日期字段格式转化


In [4]:
train_df['record_date'] = pd.to_datetime(train_df['record_date'], format='mixed')

In [5]:
train_df.head()

,user_id,record_date,power_consumption
0,1,2015-01-01,1135.0
1,1,2015-01-02,570.0
2,1,2015-01-03,3418.0
3,1,2015-01-04,3968.0
4,1,2015-01-05,3986.0


### 数据汇总聚合
最终的目标是对每一天的用电量去进行预测，给定的数据是按小时维度拆分的，我们有不同的处理方式，简单的处理方式是对数据按天做一个汇总，用pandas做相关操作如下：

In [6]:
train_df = train_df[['record_date','power_consumption']].groupby(by='record_date').agg('sum')
train_df = train_df.reset_index()
train_df.head()

,record_date,power_consumption
0,2015-01-01,2900575.0
1,2015-01-02,3158211.0
2,2015-01-03,3596487.0
3,2015-01-04,3939672.0
4,2015-01-05,4101790.0


### 添加测试数据


In [7]:
test_df = pd.date_range('2016-10-1', periods=31, freq='D')
test_df = pd.DataFrame(test_df)
test_df.columns = ['record_date']
test_df['power_consumption'] = 0

In [8]:
test_df.head()

,record_date,power_consumption
0,2016-10-01,0
1,2016-10-02,0
2,2016-10-03,0
3,2016-10-04,0
4,2016-10-05,0


### 拼成总体数据


In [9]:
total_df = pd.concat([train_df, test_df])

In [10]:
total_df.tail()

,record_date,power_consumption
26,2016-10-27,0.0
27,2016-10-28,0.0
28,2016-10-29,0.0
29,2016-10-30,0.0
30,2016-10-31,0.0


## 数据探索分析与特征工程

### 构造和时间相关的强特征


In [11]:
total_df['day_of_week'] = total_df['record_date'].apply(lambda x: x.dayofweek)
total_df['day_of_month'] = total_df['record_date'].apply(lambda x: x.day)
total_df['day_of_year'] = total_df['record_date'].apply(lambda x: x.dayofyear)
total_df['month_of_year'] = total_df['record_date'].apply(lambda x: x.month)
total_df['year'] = total_df['record_date'].apply(lambda x: x.year)
total_df.head()

,record_date,power_consumption,day_of_week,day_of_month,day_of_year,month_of_year,year
0,2015-01-01,2900575.0,3,1,1,1,2015
1,2015-01-02,3158211.0,4,2,2,1,2015
2,2015-01-03,3596487.0,5,3,3,1,2015
3,2015-01-04,3939672.0,6,4,4,1,2015
4,2015-01-05,4101790.0,0,5,5,1,2015


### 添加周末特征信息

In [12]:
total_df['holiday'] =0
total_df['holiday_sat'] =0
total_df['holiday_sun'] =0

total_df.loc[total_df.day_of_week==5,'holiday'] = 1
total_df.loc[total_df.day_of_week==5,'holiday_sat'] = 1

total_df.loc[total_df.day_of_week==6,'holiday'] = 1
total_df.loc[total_df.day_of_week==6,'holiday_sun'] = 1

total_df.head()

,record_date,power_consumption,day_of_week,day_of_month,day_of_year,month_of_year,year,holiday,holiday_sat,holiday_sun
0,2015-01-01,2900575.0,3,1,1,1,2015,0,0,0
1,2015-01-02,3158211.0,4,2,2,1,2015,0,0,0
2,2015-01-03,3596487.0,5,3,3,1,2015,1,1,0
3,2015-01-04,3939672.0,6,4,4,1,2015,1,0,1
4,2015-01-05,4101790.0,0,5,5,1,2015,0,0,0


### 添加一个月的周信息

In [13]:
def week_of_month(day):
    if day in range(1,8): return 1
    if day in range(8,15): return 2
    if day in range(15,22): return 3
    if day in range(22,32): return 4
    if day in range(29,32): return 5

total_df['week_of_month'] = total_df['day_of_month'].apply(lambda x:week_of_month(x))

total_df.head()

,record_date,power_consumption,day_of_week,day_of_month,day_of_year,month_of_year,year,holiday,holiday_sat,holiday_sun,week_of_month
0,2015-01-01,2900575.0,3,1,1,1,2015,0,0,0,1
1,2015-01-02,3158211.0,4,2,2,1,2015,0,0,0,1
2,2015-01-03,3596487.0,5,3,3,1,2015,1,1,0,1
3,2015-01-04,3939672.0,6,4,4,1,2015,1,0,1,1
4,2015-01-05,4101790.0,0,5,5,1,2015,0,0,0,1


### 添加上中下旬信息

In [14]:
def period_of_month(day):
    if day in range(1,11): return 1
    if day in range(11,21): return 2
    if day in range(21,32): return 3

total_df['period_of_month'] = total_df['day_of_month'].apply(lambda x:period_of_month(x))

total_df.head()

,record_date,power_consumption,day_of_week,day_of_month,day_of_year,month_of_year,year,holiday,holiday_sat,holiday_sun,week_of_month,period_of_month
0,2015-01-01,2900575.0,3,1,1,1,2015,0,0,0,1,1
1,2015-01-02,3158211.0,4,2,2,1,2015,0,0,0,1,1
2,2015-01-03,3596487.0,5,3,3,1,2015,1,1,0,1,1
3,2015-01-04,3939672.0,6,4,4,1,2015,1,0,1,1,1
4,2015-01-05,4101790.0,0,5,5,1,2015,0,0,0,1,1


### 添加上半月下半月信息

In [15]:
def period2_of_month(day):
    if day in range(1,16): return 1
    if day in range(16,32): return 2

total_df['period2_of_month'] = total_df['day_of_month'].apply(lambda x:period2_of_month(x))

total_df.head()

,record_date,power_consumption,day_of_week,day_of_month,day_of_year,month_of_year,year,holiday,holiday_sat,holiday_sun,week_of_month,period_of_month,period2_of_month
0,2015-01-01,2900575.0,3,1,1,1,2015,0,0,0,1,1,1
1,2015-01-02,3158211.0,4,2,2,1,2015,0,0,0,1,1,1
2,2015-01-03,3596487.0,5,3,3,1,2015,1,1,0,1,1,1
3,2015-01-04,3939672.0,6,4,4,1,2015,1,0,1,1,1,1
4,2015-01-05,4101790.0,0,5,5,1,2015,0,0,0,1,1,1


### 手动填充节日信息

In [16]:
total_df['festival_pc'] = 0
total_df['festival'] = 0

total_df.loc[total_df.record_date=='2016-10-01','festival_pc'] = total_df.loc[total_df.record_date=='2015-10-01','power_consumption'].values
total_df.loc[total_df.record_date=='2016-10-01','festival'] = 1
total_df.loc[total_df.record_date=='2016-10-01','holiday'] = 1
total_df.loc[total_df.record_date=='2016-10-02','festival_pc'] = total_df.loc[total_df.record_date=='2015-10-02','power_consumption'].values
total_df.loc[total_df.record_date=='2016-10-02','festival'] = 1
total_df.loc[total_df.record_date=='2016-10-02','holiday'] = 1
total_df.loc[total_df.record_date=='2016-10-03','festival_pc'] = total_df.loc[total_df.record_date=='2015-10-03','power_consumption'].values
total_df.loc[total_df.record_date=='2016-10-03','festival'] = 1
total_df.loc[total_df.record_date=='2016-10-03','holiday'] = 1
total_df.loc[total_df.record_date=='2016-10-04','festival_pc'] = total_df.loc[total_df.record_date=='2015-10-04','power_consumption'].values
total_df.loc[total_df.record_date=='2016-10-04','festival'] = 1
total_df.loc[total_df.record_date=='2016-10-04','holiday'] = 1
total_df.loc[total_df.record_date=='2016-10-05','festival_pc'] = total_df.loc[total_df.record_date=='2015-10-05','power_consumption'].values
total_df.loc[total_df.record_date=='2016-10-05','festival'] = 1
total_df.loc[total_df.record_date=='2016-10-05','holiday'] = 1
total_df.loc[total_df.record_date=='2016-10-06','festival_pc'] = total_df.loc[total_df.record_date=='2015-10-06','power_consumption'].values
total_df.loc[total_df.record_date=='2016-10-06','festival'] = 1
total_df.loc[total_df.record_date=='2016-10-06','holiday'] = 1
total_df.loc[total_df.record_date=='2016-10-07','festival_pc'] = total_df.loc[total_df.record_date=='2015-10-07','power_consumption'].values
total_df.loc[total_df.record_date=='2016-10-07','festival'] = 1
total_df.loc[total_df.record_date=='2016-10-07','holiday'] = 1

total_df.head()

,record_date,power_consumption,day_of_week,day_of_month,day_of_year,month_of_year,year,holiday,holiday_sat,holiday_sun,week_of_month,period_of_month,period2_of_month,festival_pc,festival
0,2015-01-01,2900575.0,3,1,1,1,2015,0,0,0,1,1,1,0,0
1,2015-01-02,3158211.0,4,2,2,1,2015,0,0,0,1,1,1,0,0
2,2015-01-03,3596487.0,5,3,3,1,2015,1,1,0,1,1,1,0,0
3,2015-01-04,3939672.0,6,4,4,1,2015,1,0,1,1,1,1,0,0
4,2015-01-05,4101790.0,0,5,5,1,2015,0,0,0,1,1,1,0,0


In [17]:
col_names = total_df.columns.values
col_names

array(['record_date', 'power_consumption', 'day_of_week', 'day_of_month',
       'day_of_year', 'month_of_year', 'year', 'holiday', 'holiday_sat',
       'holiday_sun', 'week_of_month', 'period_of_month',
       'period2_of_month', 'festival_pc', 'festival'], dtype=object)

In [18]:
col_names = total_df.columns.values
counts={}
for name in col_names:
    count = total_df[name].isnull().sum()
    counts[name] = [count]
pd.DataFrame(counts)

,record_date,power_consumption,day_of_week,day_of_month,day_of_year,month_of_year,year,holiday,holiday_sat,holiday_sun,week_of_month,period_of_month,period2_of_month,festival_pc,festival
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [19]:
var_to_encode = [ 'day_of_week', 'day_of_month', 'day_of_year', 'month_of_year','holiday','holiday_sat','holiday_sun','week_of_month','period_of_month','period2_of_month','festival']
dummy_df = pd.get_dummies(total_df, columns=var_to_encode)
total_df = pd.concat([total_df[[ 'day_of_week', 'day_of_month', 'day_of_year', 'month_of_year','holiday','holiday_sat','holiday_sun','week_of_month','period_of_month','period2_of_month','festival']], dummy_df], axis=1)
total_df.head()

,day_of_week,day_of_month,day_of_year,month_of_year,holiday,holiday_sat,holiday_sun,week_of_month,period_of_month,period2_of_month,...,week_of_month_2,week_of_month_3,week_of_month_4,period_of_month_1,period_of_month_2,period_of_month_3,period2_of_month_1,period2_of_month_2,festival_0,festival_1
0,3,1,1,1,0,0,0,1,1,1,...,False,False,False,True,False,False,True,False,True,False
1,4,2,2,1,0,0,0,1,1,1,...,False,False,False,True,False,False,True,False,True,False
2,5,3,3,1,1,1,0,1,1,1,...,False,False,False,True,False,False,True,False,True,False
3,6,4,4,1,1,0,1,1,1,1,...,False,False,False,True,False,False,True,False,True,False
4,0,5,5,1,0,0,0,1,1,1,...,False,False,False,True,False,False,True,False,True,False


In [20]:
linear_model_df = total_df.drop([ 'day_of_week', 'day_of_month', 'day_of_year','holiday','holiday_sat','holiday_sun','week_of_month','period_of_month','period2_of_month','festival'], axis=1)
train_X = linear_model_df[~((linear_model_df.year==2016)&(linear_model_df.month_of_year==10))]
test_X = linear_model_df[((linear_model_df.year==2016)&(linear_model_df.month_of_year==10))]
train_y = train_X.power_consumption
test_y = test_X.power_consumption
train_X = train_X.drop(['power_consumption','record_date','month_of_year','year'],axis=1)
test_X = test_X.drop(['power_consumption','record_date','month_of_year','year'],axis=1)

from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV

linear_regression =RidgeCV(alphas=[0.25, 0.5, 0.7], cv=10)
linear_regression.fit(train_X, train_y)

linear_regression.score(train_X, train_y)

0.6801827719053649

In [21]:
from datetime import datetime

#完成提交日期格式的转换
def dataprocess(t):
    t = str(t)[0:10]
    time = datetime.strptime(t, '%Y-%m-%d')
    res = time.strftime('%Y%m%d')
    return res

#生成10月份31天的时间段
commit_df = pd.date_range('2016/10/1', periods=31, freq='D')
commit_df = pd.DataFrame(commit_df)
commit_df.columns = ['predict_date']

#用模型进行预测
prediction = linear_regression.predict(test_X.values)
commit_df['predict_power_consumption'] = pd.DataFrame(prediction).astype('int')
commit_df['predict_date'] = commit_df['predict_date'].apply(dataprocess)
commit_df.head()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RidgeCV was fitted with feature names
  warnings.warn(


,predict_date,predict_power_consumption
0,20161001,2918411
1,20161002,2788930
2,20161003,3234254
3,20161004,3475415
4,20161005,3507292


In [22]:
train_X = total_df[~((total_df.year==2016)&(total_df.month_of_year==10))]
test_X = total_df[((total_df.year==2016)&(total_df.month_of_year==10))]
train_y = train_X.power_consumption
train_X = train_X.drop(['power_consumption','record_date','year'],axis=1)
test_X = test_X.drop(['power_consumption','record_date','year'],axis=1)

print('The shape of the X for training: ', train_X.shape)
train_X.head()

The shape of the X for training:  (639, 444)


,day_of_week,day_of_month,day_of_year,month_of_year,holiday,holiday_sat,holiday_sun,week_of_month,period_of_month,period2_of_month,...,week_of_month_2,week_of_month_3,week_of_month_4,period_of_month_1,period_of_month_2,period_of_month_3,period2_of_month_1,period2_of_month_2,festival_0,festival_1
0,3,1,1,1,0,0,0,1,1,1,...,False,False,False,True,False,False,True,False,True,False
1,4,2,2,1,0,0,0,1,1,1,...,False,False,False,True,False,False,True,False,True,False
2,5,3,3,1,1,1,0,1,1,1,...,False,False,False,True,False,False,True,False,True,False
3,6,4,4,1,1,0,1,1,1,1,...,False,False,False,True,False,False,True,False,True,False
4,0,5,5,1,0,0,0,1,1,1,...,False,False,False,True,False,False,True,False,True,False


In [ ]:
# 导入所需的库
import os
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

# 取出建模所用的训练数据矩阵和目标y
X = train_X.values
y = train_y.values

# 候选参数，用于参数调优
param_grid = {
              'max_depth': [3, 4, 5, 7, 9],
              'n_estimators': [20, 40, 50, 80, 100, 200, 400, 800, 1000, 1200],
              'learning_rate': [0.05, 0.1, 0.2, 0.3],
              'subsample': [0.8, 1],
              'colsample_bylevel':[0.8, 1]
             }

# 使用xgboost的regressor完成回归
xgb_model = xgb.XGBRegressor()
# 数据拟合
rgs = GridSearchCV(xgb_model, param_grid, n_jobs=8)
rgs.fit(X, y)

print(rgs.best_score_)
print(rgs.best_params_)

In [24]:
from datetime import datetime

def dataprocess(t):
    t = str(t)[0:10]
    time = datetime.strptime(t, '%Y-%m-%d')
    res = time.strftime('%Y%m%d')
    return res

commit_df = pd.date_range('2016/10/1', periods=31, freq='D')
commit_df = pd.DataFrame(commit_df)
commit_df.columns = ['predict_date']
prediction = rgs.predict(test_X.values)
commit_df['predict_power_consumption'] = pd.DataFrame(prediction).astype('int')
commit_df['predict_date'] = commit_df['predict_date'].apply(dataprocess)

commit_df.head()

NotFittedError: This GridSearchCV instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=20, subsample=1, learning_rate=0.3, max_depth=3, colsample_bylevel=0.8)
xgb_model.fit(train_X, train_y)
xgb_model.feature_importances_

In [ ]:

# 树状模型建模：分离训练集和测试集
train_X = total_df[~((total_df.year==2016)&(total_df.month_of_year==10))]
test_X = total_df[((total_df.year==2016)&(total_df.month_of_year==10))]
train_y = train_X.power_consumption
train_X = train_X.drop(['power_consumption','record_date','year'],axis=1)
test_X = test_X.drop(['power_consumption','record_date','year'],axis=1)

print('The shape of the X for training: ', train_X.shape)
train_X.head()

In [ ]:
import matplotlib.pyplot as plt
plt.bar([0,1,2,3,4,5,6], total_df.groupby('day_of_week').power_consumption.sum())

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
plt.plot(total_df.record_date, total_df['power_consumption'])
plt.ylabel('power_consumption')
plt.show()

In [ ]:
# 导入所需的库
import os
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=20, subsample=1, learning_rate=0.3, max_depth=3, colsample_bylevel=0.8)
xgb_model.fit(train_X, train_y)

In [ ]:
print("Feature ranking:")
feature_names = [u'day_of_week', u'day_of_month', u'day_of_year', u'month_of_year',
       u'holiday', u'holiday_sat', u'holiday_sun', u'week_of_month',
       u'period_of_month', u'period2_of_month', u'festival_pc', u'festival']
feature_importances = xgb_model.feature_importances_
indices = np.argsort(feature_importances)[::-1]

for f in indices:
    print("feature %s (%f)" % (feature_names[f], feature_importances[f]))

plt.figure(figsize=(20,8))
plt.title("Feature importances")
plt.bar(range(len(feature_importances)), feature_importances[indices],
       color="b",align="center")
plt.xticks(range(len(feature_importances)), np.array(feature_names)[indices])
plt.xlim([-1, train_X.shape[1]])
plt.show()

In [ ]:
xgb.plot_tree(xgb_model, num_trees=10)
fig = plt.gcf()
fig.set_size_inches(150, 100)